# Les files — Exercice 7 : les nombres de Hamming

**Notions travaillées :** décomposition en facteurs premiers, force brute contre algorithme astucieux, fusion de files triées, optimisation.

Un **nombre de Hamming** est un entier `n ≥ 1` dont les facteurs premiers sont tous **2, 3 ou 5**, c'est-à-dire de la forme `2^a × 3^b × 5^c`.
- `60 = 2² × 3 × 5` est un nombre de Hamming ;
- `14 = 2 × 7` n'en est pas un (facteur 7) ;
- `1 = 2⁰ × 3⁰ × 5⁰` en est un.

(L'énoncé écrit deux fois *Hammnig* : c'est une faute de frappe pour *Hamming*. On nomme la fonction de la partie A `hamming_fb`.)

## Mise en place

On importe la classe `File` écrite à l'exercice 1 (le fichier `file.py` doit être **dans le même dossier** que ce notebook).

L'énoncé interdit d'utiliser le type `list` : on ne manipule les files **qu'avec leurs méthodes** `enfile`, `defile`, `taille`, `est_vide`. La petite fonction `file_de` sert uniquement à fabriquer rapidement des files pour les tests.

On affiche une file sous la forme `tête < ... < queue` : on **défile à gauche** et on **enfile à droite**.

In [1]:
from file import File   # la classe File de l'exercice 1 (fichier file.py placé à côté du notebook)


def file_de(*elements):
    """Outil pour les tests : construit une file contenant les éléments donnés (le premier en tête)."""
    f = File()
    for el in elements:
        f.enfile(el)
    return f


print(file_de("a", "b", "c"))


# la liste de l'énoncé, uniquement pour vérifier nos résultats
TRENTE_PREMIERS = file_de(1, 2, 3, 4, 5, 6, 8, 9, 10, 12, 15, 16, 18, 20, 24, 25, 27, 30,
                          32, 36, 40, 45, 48, 50, 54, 60, 64, 72, 75, 80)

tête < 'a' < 'b' < 'c' < queue


---
## Partie A : la force brute

### 1. `is_hamming(n)`

**Idée :** on enlève de `n` tous ses facteurs 2, puis tous ses facteurs 3, puis tous ses facteurs 5. S'il ne reste que **1**, il n'y avait pas d'autre facteur premier.

Exemple : `60 → 30 → 15` (plus divisible par 2) `→ 5` (plus par 3) `→ 1` ✔. Et `14 → 7` : il reste 7 ≠ 1 ✘.

⚠️ Il faut un `while` et non un `if` : `8 = 2³` doit être divisé **trois fois** par 2.

⚠️ Il faut **la division entière** `//` : avec `/`, on obtient des flottants (`30.0`).

In [2]:
def is_hamming(n):
    if n < 1:
        return False
    for p in (2, 3, 5):
        while n % p == 0:
            n = n // p
    return n == 1


assert is_hamming(1) and is_hamming(60) and is_hamming(80)
assert not is_hamming(14) and not is_hamming(7) and not is_hamming(0)

### 2. `hamming_fb(n)`

On parcourt les entiers `k = 1, 2, 3, …` et on enfile ceux qui sont des nombres de Hamming, **tant que** la file ne contient pas `n` éléments. On ne sait pas à l'avance jusqu'où aller : c'est donc un `while`, pas un `for`.

In [3]:
def hamming_fb(n):
    resultat = File()
    k = 1
    while resultat.taille() < n:
        if is_hamming(k):
            resultat.enfile(k)
        k += 1
    return resultat


print(hamming_fb(30))
assert repr(hamming_fb(30)) == repr(TRENTE_PREMIERS)

tête < 1 < 2 < 3 < 4 < 5 < 6 < 8 < 9 < 10 < 12 < 15 < 16 < 18 < 20 < 24 < 25 < 27 < 30 < 32 < 36 < 40 < 45 < 48 < 50 < 54 < 60 < 64 < 72 < 75 < 80 < queue


💡 **Le défaut de la force brute :** les nombres de Hamming deviennent **très rares**. Le 1000ᵉ vaut 51 200 000 : il faudrait tester plus de 51 millions d'entiers pour en trouver 1000. On le constate en mesurant le temps de calcul :

In [4]:
from time import perf_counter

for n in [100, 200, 400]:
    debut = perf_counter()
    f = hamming_fb(n)
    for _ in range(n - 1):
        f.defile()
    print(f"hamming_fb({n}) : {perf_counter() - debut:.3f} s, le {n}e nombre de Hamming vaut {f.defile()}")

hamming_fb(100) : 0.001 s, le 100e nombre de Hamming vaut 1536
hamming_fb(200) : 0.005 s, le 200e nombre de Hamming vaut 16200
hamming_fb(400) : 0.079 s, le 400e nombre de Hamming vaut 311040


Quand `n` double, le temps est multiplié bien plus que par 2 : la force brute ne passe pas à l'échelle.

---
## Partie B : avec trois files

**L'idée :** au lieu de chercher les nombres de Hamming parmi tous les entiers, on les **fabrique** : tout nombre de Hamming autre que 1 est le double, le triple ou le quintuple d'un nombre de Hamming plus petit.

### 1. `min3(a, b, c)`

On compare deux par deux. (La fonction `min` de python le fait déjà, mais l'énoncé demande de l'écrire.)

In [5]:
def min3(a, b, c):
    m = a
    if b < m:
        m = b
    if c < m:
        m = c
    return m


assert min3(1, 2, 3) == 1 and min3(3, 1, 2) == 1 and min3(3, 2, 1) == 1 and min3(4, 4, 4) == 4

### 2. `head(f)`

On veut **lire** la tête sans la retirer. La classe `File` n'a pas de méthode pour ça : on fait un **tour complet** (rotation), en mémorisant le **premier** élément défilé.

⚠️ Faire `x = f.defile()` puis `f.enfile(x)` ne suffit pas : la tête passerait **en queue**. Il faut aller jusqu'au bout du tour pour qu'elle revienne devant.

Si la file est vide, il n'y a pas de tête : on déclenche une erreur, comme `defile`.

In [6]:
def head(f):
    if f.est_vide():
        raise IndexError("head() sur une file vide")
    tete = f.defile()
    f.enfile(tete)
    for _ in range(f.taille() - 1):    # on termine le tour : tete revient devant
        f.enfile(f.defile())
    return tete


f = file_de(7, 8, 9)
assert head(f) == 7
assert repr(f) == repr(file_de(7, 8, 9))     # file inchangée

### 3. `hamming(n)`

On applique l'algorithme de l'énoncé. Les trois files contiennent au départ le nombre 1.

**Pourquoi le plus petit `m` est-il toujours en tête ?** Chaque file est **triée** dans l'ordre croissant : on y enfile `2m`, `3m` ou `5m` avec des `m` de plus en plus grands. Le plus petit élément de chaque file est donc sa tête, et le plus petit élément des trois files est le plus petit des trois têtes. Pour la même raison, « retirer `m` de chacune des files le contenant » revient à le **défiler** des files dont il est la tête.

**Déroulement des premiers tours :**

| `m` | résultats | `f2` | `f3` | `f5` |
|:---|:---|:---|:---|:---|
| départ | | `1` | `1` | `1` |
| 1 | `1` | `2` | `3` | `5` |
| 2 | `1 2` | `4` | `3 6` | `5 10` |
| 3 | `1 2 3` | `4 6` | `6 9` | `5 10 15` |
| 4 | `1 2 3 4` | `6 8` | `6 9 12` | `5 10 15 20` |
| 5 | `1 2 3 4 5` | `6 8 10` | `6 9 12 15` | `10 15 20 25` |
| 6 | `1 2 3 4 5 6` | `8 10 12` | `9 12 15 18` | `10 15 20 25 30` |

Au tour de `m = 6`, le nombre 6 est en tête de `f2` **et** de `f3` : on le défile des deux, sinon il apparaîtrait deux fois dans les résultats.

Le paramètre `afficher` servira à la partie C.

In [7]:
def hamming(n, afficher=False):
    resultat = File()
    f2, f3, f5 = File(), File(), File()
    f2.enfile(1)
    f3.enfile(1)
    f5.enfile(1)
    while resultat.taille() < n:
        m = min3(head(f2), head(f3), head(f5))
        resultat.enfile(m)
        if head(f2) == m:
            f2.defile()
        if head(f3) == m:
            f3.defile()
        if head(f5) == m:
            f5.defile()
        f2.enfile(2 * m)
        f3.enfile(3 * m)
        f5.enfile(5 * m)
    if afficher:
        print("f2 :", f2)
        print("f3 :", f3)
        print("f5 :", f5)
    return resultat


print(hamming(30))
assert repr(hamming(30)) == repr(TRENTE_PREMIERS)
assert repr(hamming(300)) == repr(hamming_fb(300))

tête < 1 < 2 < 3 < 4 < 5 < 6 < 8 < 9 < 10 < 12 < 15 < 16 < 18 < 20 < 24 < 25 < 27 < 30 < 32 < 36 < 40 < 45 < 48 < 50 < 54 < 60 < 64 < 72 < 75 < 80 < queue


⚠️ **Erreur fréquente :** utiliser `if / elif / elif` pour défiler. On ne retirerait `m` que d'**une** file, et un nombre comme 6 apparaîtrait deux fois dans les résultats.

---
## Partie C : optimisation

### 1. L'état des files pour `hamming(20)`

In [8]:
res = hamming(20, afficher=True)

f2 : tête < 40 < 48 < 50 < 54 < 60 < 64 < 72 < queue
f3 : tête < 45 < 48 < 54 < 60 < 72 < 75 < 81 < 90 < 96 < 108 < queue
f5 : tête < 40 < 45 < 50 < 60 < 75 < 80 < 90 < 100 < 120 < 125 < 135 < 150 < 160 < 180 < queue


On constate que des nombres apparaissent dans plusieurs files : par exemple **60** est dans les trois (60 = 2 × 30 = 3 × 20 = 5 × 12), et beaucoup de nombres sont dans deux files (45 = 3 × 15 = 5 × 9 est dans `f3` et `f5`, 48 = 2 × 24 = 3 × 16 est dans `f2` et `f3`…). Comptons-les avec `est_dans_la_file` :

In [9]:
def est_dans_la_file(f, val):
    trouve = False
    for _ in range(f.taille()):
        el = f.defile()
        if el == val:
            trouve = True
        f.enfile(el)
    return trouve


def files_finales(n):
    """Rejoue hamming(n) et renvoie les trois files f2, f3, f5 finales."""
    f2, f3, f5 = File(), File(), File()
    f2.enfile(1)
    f3.enfile(1)
    f5.enfile(1)
    for _ in range(n):
        m = min3(head(f2), head(f3), head(f5))
        for f in (f2, f3, f5):
            if head(f) == m:
                f.defile()
        f2.enfile(2 * m)
        f3.enfile(3 * m)
        f5.enfile(5 * m)
    return f2, f3, f5


f2, f3, f5 = files_finales(20)
for k in range(1, 401):
    nb_files = est_dans_la_file(f2, k) + est_dans_la_file(f3, k) + est_dans_la_file(f5, k)
    if nb_files >= 2:
        print(k, "est dans", nb_files, "files")

40 est dans 2 files
45 est dans 2 files
48 est dans 2 files
50 est dans 2 files
54 est dans 2 files
60 est dans 3 files
72 est dans 2 files
75 est dans 2 files
90 est dans 2 files


Ces doublons ne faussent pas le résultat (on les défile ensemble), mais ils font **grossir les files inutilement** et coûtent des comparaisons.

### 2. Insérer chaque nombre dans une seule file

**L'idée :** un même nombre peut être fabriqué de plusieurs façons (60 = 2 × 30 = 3 × 20 = 5 × 12). On décide d'une **seule façon officielle** de fabriquer chaque nombre :
on écrit `x = 2^a × 3^b × 5^c` en multipliant **d'abord par les 2, puis par les 3, puis par les 5**.

Concrètement, on note de quelle file vient `m` :
- si `m` vient de **`f2`** (il ne contient que des facteurs 2) : on peut encore multiplier par 2, 3 ou 5 → on enfile `2m` dans `f2`, `3m` dans `f3`, `5m` dans `f5` ;
- si `m` vient de **`f3`** (il contient au moins un facteur 3, pas de 5) : on n'a plus le droit de multiplier par 2 → on enfile seulement `3m` dans `f3` et `5m` dans `f5` ;
- si `m` vient de **`f5`** (il contient au moins un facteur 5) : on ne peut plus multiplier que par 5 → on enfile seulement `5m` dans `f5`.

Ainsi, chaque nombre `2^a × 3^b × 5^c` n'est fabriqué que d'**une** façon, donc inséré dans **une seule** file. Les files restent triées, et `m` n'est plus en tête que d'une seule file : un `if / elif / else` suffit maintenant.

**Le départ :** on ne met plus 1 dans les trois files (ce serait déjà un doublon). On place 1 dans les résultats et on « fabrique » ses multiples : `f2 = 2`, `f3 = 3`, `f5 = 5`.

| `m` | vient de | on enfile | `f2` | `f3` | `f5` |
|:---|:---|:---|:---|:---|:---|
| 1 | départ | 2, 3, 5 | `2` | `3` | `5` |
| 2 | `f2` | 4, 6, 10 | `4` | `3 6` | `5 10` |
| 3 | `f3` | 9, 15 | `4` | `6 9` | `5 10 15` |
| 4 | `f2` | 8, 12, 20 | `8` | `6 9 12` | `5 10 15 20` |
| 5 | `f5` | 25 | `8` | `6 9 12` | `10 15 20 25` |
| 6 | `f3` | 18, 30 | `8` | `9 12 18` | `10 15 20 25 30` |

In [10]:
def hamming_opt(n, afficher=False):
    resultat = File()
    f2, f3, f5 = File(), File(), File()
    if n >= 1:
        resultat.enfile(1)
        f2.enfile(2)
        f3.enfile(3)
        f5.enfile(5)
    while resultat.taille() < n:
        h2, h3, h5 = head(f2), head(f3), head(f5)
        m = min3(h2, h3, h5)
        resultat.enfile(m)
        if m == h2:                 # m = 2^a
            f2.defile()
            f2.enfile(2 * m)
            f3.enfile(3 * m)
            f5.enfile(5 * m)
        elif m == h3:               # m = 2^a × 3^b avec b ≥ 1
            f3.defile()
            f3.enfile(3 * m)
            f5.enfile(5 * m)
        else:                       # m contient au moins un facteur 5
            f5.defile()
            f5.enfile(5 * m)
    if afficher:
        print("f2 :", f2)
        print("f3 :", f3)
        print("f5 :", f5)
    return resultat


res = hamming_opt(20, afficher=True)

f2 : tête < 64 < queue
f3 : tête < 48 < 54 < 72 < 81 < 96 < 108 < queue
f5 : tête < 40 < 45 < 50 < 60 < 75 < 80 < 90 < 100 < 120 < 125 < 135 < 150 < 160 < 180 < queue


Plus aucun nombre n'apparaît dans deux files, et les files sont bien plus courtes. On vérifie que le résultat est **le même** qu'avec la partie B :

In [11]:
assert repr(hamming_opt(30)) == repr(TRENTE_PREMIERS)
assert repr(hamming_opt(1)) == repr(file_de(1))
assert hamming_opt(0).est_vide()
for n in [50, 200, 1000]:
    assert repr(hamming_opt(n)) == repr(hamming(n))
print("Tous les tests passent ✔")

for n in [200, 1000]:
    debut = perf_counter()
    hamming(n)
    t_b = perf_counter() - debut
    debut = perf_counter()
    hamming_opt(n)
    t_c = perf_counter() - debut
    print(f"n = {n} : partie B {t_b:.3f} s, partie C {t_c:.3f} s")

Tous les tests passent ✔
n = 200 : partie B 0.005 s, partie C 0.002 s


n = 1000 : partie B 0.075 s, partie C 0.021 s


💡 **Bilan :**
- la **force brute** (A) teste tous les entiers : elle devient vite inutilisable ;
- l'algorithme à **trois files** (B) ne fabrique que des nombres de Hamming : c'est le grand gain ;
- l'**optimisation** (C) évite les doublons dans les files, qui sont plus courtes : chaque `head`, qui fait un tour complet de la file, coûte moins cher.

C'est un exemple célèbre, proposé par le mathématicien Edsger Dijkstra pour montrer l'intérêt des files.